# Lumen — QLoRA Fine-tune of Llama 3.1 8B
Run on **T4** in Colab. Targets ~3-4 hours, ~30-40 credits.

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets peft trl bitsandbytes accelerate

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Config — adjust paths if needed
DATASET_PATH = '/content/drive/MyDrive/Lumen/merged_121.jsonl'  # 1.2.1 mixed dataset
OUTPUT_DIR   = '/content/drive/MyDrive/Lumen/lumen-121-checkpoints'
FINAL_MODEL  = '/content/drive/MyDrive/Lumen/lumen-121-merged'

BASE_MODEL   = 'meta-llama/Meta-Llama-3.1-8B-Instruct'
HF_TOKEN     = ''  # paste your HuggingFace token here (needs llama access)

In [ ]:
# 4. Load dataset
import json
from datasets import Dataset

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

raw = load_jsonl(DATASET_PATH)
dataset = Dataset.from_list(raw)
dataset = dataset.train_test_split(test_size=0.01, seed=42)
print(f'Train: {len(dataset["train"])}  Val: {len(dataset["test"])}')

In [ ]:
# 5. Load tokenizer + model in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
model.config.use_cache = False
print('Model loaded.')

In [ ]:
# 6. Apply LoRA
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# 7. Format dataset into chat template
def format_example(example):
    return {'text': tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False
    )}

dataset = dataset.map(format_example)
print(dataset['train'][0]['text'][:500])

In [ ]:
# 8. Train
from trl import SFTTrainer, SFTConfig

# Lumen 1.2.1 — mixed coding + reasoning dataset
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,   # effective batch = 16
    gradient_checkpointing=True,
    learning_rate=1.5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=800,
    fp16=True,
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    eval_strategy="steps",
    eval_steps=500,
    load_best_model_at_end=True,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
    max_seq_length=2048,
)

trainer.train()

In [ ]:
# 9. Merge LoRA weights into base model and save
from peft import PeftModel
from transformers import AutoModelForCausalLM
import torch

print('Merging LoRA weights...')
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map='cpu',
    token=HF_TOKEN,
)
merged = PeftModel.from_pretrained(base, OUTPUT_DIR)
merged = merged.merge_and_unload()
merged.save_pretrained(FINAL_MODEL)
tokenizer.save_pretrained(FINAL_MODEL)
print(f'Saved merged model to {FINAL_MODEL}')

In [ ]:
# 10. Quick test
from transformers import pipeline

pipe = pipeline('text-generation', model=FINAL_MODEL, tokenizer=tokenizer,
                torch_dtype=torch.float16, device_map='auto')

messages = [{'role': 'user', 'content': 'How do I listen for player join events in Paper?'}]
out = pipe(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True),
           max_new_tokens=300, do_sample=False)
print(out[0]['generated_text'])